# 01. Home Credit Default Risk data preparation

**The unit of analysis is one current application, `SK_ID_CURR`.** This notebook audits the data, runs a technical pilot and performs full deterministic preparation. No models are trained here.

Basis: the local `Aubakirov_Dagystanov_A2_Methods_v2.pdf`, Methods and Materials — Assignment 2, especially page 2. The accepted plan is **7 models × 4 levels × 5 repetitions = 140 main training runs**. Each of the 28 model–level pairs has a separate Randomized Search: 20 configurations × stratified 3-fold CV, selected by log-loss; the parameters are then fixed for five repetitions. This adds 1 680 candidate CV fits to the 140 main fits, excluding internal ensemble training, refit and calibration.

Data-fitted transformations—medians, scaling, one-hot or other learned encoding, and feature selection—**remain inside the next-stage Pipeline**. No `.fit()`, class resampling or TARGET-based feature selection is performed on Home Credit here. Future Pipeline behavior is checked only with synthetic data.

Implementation is in `src/data_audit.py` and `src/data_preparation.py`; the notebook explicitly runs each stage and saves its actual results. CSV inputs remain unchanged. Dependencies are pinned in `requirements.txt` (`python -m pip install -r requirements.txt`); from the project root, use `python scripts/execute_notebook.py` or Jupyter Run All.

In [1]:
from pathlib import Path
import sys, os, time, json, gc, shutil
import pandas as pd
import psutil
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/data_preparation.py').is_file(), 'Run the notebook from the project root or notebooks/'
sys.path.insert(0, str(ROOT))
from src.data_audit import audit_sources
from src.data_preparation import (source_manifest, connect, run_preparation,
                                  compare_pilot_to_full, dump_json)

DATA_DIR = Path(os.environ.get('HOME_CREDIT_DATA_DIR', r'C:\Users\askat\Study\RMT\home-credit-default-risk'))
REPORTS = ROOT / 'reports'
PILOT_DIR = ROOT / 'data/processed/pilot'
FULL_DIR = ROOT / 'data/processed/full'
PILOT_SIZE = 3000
RUN_FULL = True
RUN_START = time.perf_counter()
REPORTS.mkdir(exist_ok=True)
print('Source CSV files:', DATA_DIR)
print('Total/available RAM, GiB:', round(psutil.virtual_memory().total/1024**3,2),
      round(psutil.virtual_memory().available/1024**3,2))
print('Free disk, GiB:', round(shutil.disk_usage(ROOT).free/1024**3,2))
print('DuckDB: memory_limit=2GB; threads=2; sequential sources, disk spill.')

Source CSV files: C:\Users\askat\Study\RMT\home-credit-default-risk
Total/available RAM, GiB: 15.82 6.58
Free disk, GiB: 112.7
DuckDB: memory_limit=2GB; threads=2; sequential sources, disk spill.


## 1. File availability and source integrity

All eight required files are checked. Execution stops and names any missing file. A SHA-256 is saved for each source and recomputed after preparation. `application_test.csv` and `sample_submission.csv` are outside the labelled study population and are not used.

Raw CSV, Parquet and temporary files are excluded through `.gitignore`. Only code, notebooks and small de-identified summaries belong in Git.

In [2]:
manifest_before = source_manifest(DATA_DIR)
dump_json(REPORTS / 'source_sha256_before.json', manifest_before)
display(pd.DataFrame(manifest_before)[['file','bytes']])

,file,bytes
0,application_train.csv,166133370
1,bureau.csv,170016717
2,bureau_balance.csv,375592889
3,POS_CASH_balance.csv,392703158
4,credit_card_balance.csv,424582605
5,installments_payments.csv,723118349
6,previous_application.csv,404973293
7,HomeCredit_columns_description.csv,37383


## 2. Full audit before feature construction

The audit reads **every row of every CSV**, even when the subsequent technical pilot uses 3 000 clients. DuckDB initially infers types from 100 000 rows, then reads the full contents strictly: `ignore_errors=false`, so incompatible types raise errors. Each column records its type, missing count/fraction and numeric min/max; date fields also record positive values, zeros and the `365243` sentinel.

Exact duplicate rows and candidate keys are checked. Repeated payment keys may represent partial payments and are not automatically removed. Unique application, bureau and previous_application keys are required for joins. Histories without a labelled current application are counted in the audit but excluded from X.

The dictionary is read as `cp1252`, with whitespace trimmed from column names. Documented `SK_BUREAU_ID` is matched to actual `SK_ID_BUREAU`. Fields described in the dictionary but absent from CSV are not invented. Relationship checks distinguish missing parents, incorrect contract owners and clients outside `application_train`.

In [3]:
audit_connection = connect(ROOT / 'tmp/audit_spill')
try:
    audit = audit_sources(audit_connection, DATA_DIR, REPORTS / 'audit')
finally:
    audit_connection.close()
table_audit = pd.read_csv(REPORTS / 'audit/table_summary.csv')
display(table_audit)
display(pd.read_csv(REPORTS / 'audit/target_balance.csv'))
key_tables = table_audit[table_audit.table.isin(['application_train.csv','bureau.csv','previous_application.csv'])]
assert (key_tables.key_duplicate_surplus == 0).all()
assert (key_tables.rows_with_null_key == 0).all()
assert not audit['dictionary_unmatched_columns']
print('All actual columns match the dictionary; parent-table keys are unique.')

Audit application_train.csv: 307,511 rows, 122 columns; full duplicates 0


Audit bureau.csv: 1,716,428 rows, 17 columns; full duplicates 0


Audit bureau_balance.csv: 27,299,925 rows, 3 columns; full duplicates 0


Audit POS_CASH_balance.csv: 10,001,358 rows, 8 columns; full duplicates 0


Audit credit_card_balance.csv: 3,840,312 rows, 23 columns; full duplicates 0


Audit installments_payments.csv: 13,605,401 rows, 8 columns; full duplicates 0


Audit previous_application.csv: 1,670,214 rows, 37 columns; full duplicates 0


,table,rows,columns,size_bytes,exact_duplicate_rows,candidate_key,key_groups,key_duplicate_surplus,max_rows_per_key,rows_with_null_key,key_expected_unique,key_note,audit_seconds
0,application_train.csv,307511,122,166133370,0,SK_ID_CURR,307511,0,1,0,True,NaN,7.741
1,bureau.csv,1716428,17,170016717,0,SK_ID_BUREAU,1716428,0,1,0,True,NaN,3.901
2,bureau_balance.csv,27299925,3,375592889,0,"SK_ID_BUREAU, MONTHS_BALANCE",27299925,0,1,0,True,NaN,14.198
3,POS_CASH_balance.csv,10001358,8,392703158,0,"SK_ID_PREV, MONTHS_BALANCE",10001358,0,1,0,True,NaN,8.799
4,credit_card_balance.csv,3840312,23,424582605,0,"SK_ID_PREV, MONTHS_BALANCE",3840312,0,1,0,True,NaN,8.707
5,installments_payments.csv,13605401,8,723118349,0,"SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTAL...",13604631,770,2,2905,False,Partial payments can repeat this candidate...,12.975
6,previous_application.csv,1670214,37,404973293,0,SK_ID_PREV,1670214,0,1,0,True,NaN,6.844
7,HomeCredit_columns_description.csv,219,5,37383,0,"Table, Row",219,0,1,0,True,The dictionary was read using cp1252 encoding,NaN


,TARGET,rows,fraction
0,0,282686,0.919271
1,1,24825,0.080729


All actual columns match the dictionary; parent-table keys are unique.


In [4]:
column_audit = pd.read_csv(REPORTS / 'audit/column_audit.csv')
display(column_audit.sort_values('missing_fraction', ascending=False)[
    ['table','column','dtype','missing_count','missing_fraction']].head(25))
display(column_audit[column_audit.positive_count.notna()][
    ['table','column','min','max','positive_count','zero_count','placeholder_365243_count']])
relationships = pd.read_csv(REPORTS / 'audit/relationships.csv')
display(relationships)
print('Dictionary discrepancies:')
display(pd.read_csv(REPORTS / 'audit/dictionary_extra_rows.csv')[['Table','Row','Description']])

,table,column,dtype,missing_count,missing_fraction
194,previous_application.csv,RATE_INTEREST_PRIMARY,DOUBLE,1664263,0.996437
195,previous_application.csv,RATE_INTEREST_PRIVILEGED,DOUBLE,1664263,0.996437
138,bureau.csv,AMT_ANNUITY,DOUBLE,1226791,0.714735
76,application_train.csv,COMMONAREA_MEDI,DOUBLE,214865,0.698723
48,application_train.csv,COMMONAREA_AVG,DOUBLE,214865,0.698723
62,application_train.csv,COMMONAREA_MODE,DOUBLE,214865,0.698723
56,application_train.csv,NONLIVINGAPARTMENTS_AVG,DOUBLE,213514,0.694330
84,application_train.csv,NONLIVINGAPARTMENTS_MEDI,DOUBLE,213514,0.694330
70,application_train.csv,NONLIVINGAPARTMENTS_MODE,DOUBLE,213514,0.694330
86,application_train.csv,FONDKAPREMONT_MODE,VARCHAR,210295,0.683862


,table,column,min,max,positive_count,zero_count,placeholder_365243_count
17,application_train.csv,DAYS_BIRTH,-25229.0,-7489.0,0.0,0.0,0.0
18,application_train.csv,DAYS_EMPLOYED,-17912.0,365243.0,55374.0,2.0,55374.0
19,application_train.csv,DAYS_REGISTRATION,-24672.0,0.0,0.0,80.0,0.0
20,application_train.csv,DAYS_ID_PUBLISH,-7197.0,0.0,0.0,16.0,0.0
95,application_train.csv,DAYS_LAST_PHONE_CHANGE,-4292.0,0.0,0.0,37672.0,0.0
126,bureau.csv,DAYS_CREDIT,-2922.0,0.0,0.0,25.0,0.0
128,bureau.csv,DAYS_CREDIT_ENDDATE,-42060.0,31199.0,602603.0,883.0,0.0
129,bureau.csv,DAYS_ENDDATE_FACT,-42023.0,0.0,0.0,64.0,0.0
137,bureau.csv,DAYS_CREDIT_UPDATE,-41947.0,372.0,17.0,605.0,0.0
140,bureau_balance.csv,MONTHS_BALANCE,-96.0,0.0,0.0,610965.0,0.0


,table,check,rows_or_ids,interpretation
0,bureau.csv,rows_outside_labeled_applications,251103,Clients may have no published history...
1,bureau.csv,distinct_clients_outside_labeled_applications,42320,Does not imply an orphan key in the full competition...
2,bureau.csv,rows_linked_to_labeled_applications,1465325,History owned by a client in application_train
3,POS_CASH_balance.csv,rows_outside_labeled_applications,1457983,Clients may have no published history...
4,POS_CASH_balance.csv,distinct_clients_outside_labeled_applications,47808,Does not imply an orphan key in the full competition...
5,POS_CASH_balance.csv,rows_linked_to_labeled_applications,8543375,History owned by a client in application_train
6,credit_card_balance.csv,rows_outside_labeled_applications,612347,Clients may have no published history...
7,credit_card_balance.csv,distinct_clients_outside_labeled_applications,16653,Does not imply an orphan key in the full competition...
8,credit_card_balance.csv,rows_linked_to_labeled_applications,3227965,History owned by a client in application_train
9,installments_payments.csv,rows_outside_labeled_applications,2013809,Clients may have no published history...


Dictionary discrepancies:


,Table,Row,Description
0,previous_application.csv,NFLAG_MICRO_CASH,Flag Micro finance loan


## 3. Temporal availability and feature rules

The dataset has no calendar dates or complete change history. The rules below therefore provide reproducible filtering with available fields, but do not prove point-in-time availability of every attribute. Application fields, including `EXT_SOURCE_1/2/3`, are treated as a snapshot at application; the dictionary does not disclose external-score dates. Final states without a defensible availability date are excluded.

| Source | Rule and interpretation |
|---|---|
| application | `DAYS_EMPLOYED=365243` and other positive actual-date offsets become NULL; unknown employment dates receive an indicator. Age and gender are removed from X before ratios are created. |
| bureau | `DAYS_CREDIT < 0` and `DAYS_CREDIT_UPDATE < 0`: the decision and latest information must precede the application day. `DAYS_CREDIT_ENDDATE` is remaining contractual duration, so positive planned dates are retained. `DAYS_ENDDATE_FACT` is allowed only when <0. |
| bureau_balance | `MONTHS_BALANCE < 0`, linked through eligible bureau records → application. Month 0 is excluded because the exact snapshot day is unknown. |
| POS / credit card | `MONTHS_BALANCE < 0`, directly linked by `SK_ID_CURR`. The POS dictionary allows month 0 as an application-time snapshot, but the common strict rule excludes it. Future instalment counts represent scheduled obligations already known at the historical snapshot. |
| installments | Actual `DAYS_ENTRY_PAYMENT < 0`; unknown or future actual payments are excluded. Positive `DAYS_INSTALMENT` is not automatically removed: it may be the due date of a completed prepayment. `365243` is not a valid date. |
| previous | Only `DAYS_DECISION < 0`. `DAYS_FIRST_DUE` and `DAYS_LAST_DUE_1ST_VERSION` are contractual dates: positive values are retained and `365243` becomes NULL. `DAYS_FIRST_DRAWING`, `DAYS_LAST_DUE`, `DAYS_TERMINATION` are excluded from X because availability/update timing is ambiguous. |

The audit checks the `SK_ID_PREV` link to previous. Absence of a published previous application does not invalidate directly linked history with valid `SK_ID_CURR` and its own eligible date; Level 3 does not depend on the Level 4 table. All exclusions are counted in `temporal_filters.csv`; retained coverage is saved in `history_coverage.csv`.

Exact duplicates are diagnosed but not blindly removed without an event identifier. Partial `AMT_PAYMENT` values are first summed within `(SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER)`. The scheduled amount is counted once. Obligations with conflicting due dates or amounts within the key are excluded from aggregates and counted separately. Late-payment fractions describe **observed paid instalments**; entirely absent unpaid obligations cannot be reconstructed from this table.

Instalment and partial-payment amounts are aggregated as **DECIMAL(24,3)**: the actual CSV was checked to contain at most three decimal places. This avoids false underpayment caused by floating-point summation order at equality. Excess precision is checked before conversion; final features use ordinary numeric types. Lateness based on the last payment date is an observed approximation: a late overpayment may mark an obligation fully paid on time as late. It is not the exact debt-settlement date. Zero or negative ratio denominators produce NULL rather than infinity. Historical mean/min/max describe typical values and ranges; monthly-table averages are weighted by observed contract-month counts. `HAS_*` indicators denote available eligible aggregates. With no history, they equal 0, while sums, fractions and counts remain NULL.

## 4. Four nested levels and a technical pilot

1. **Level 1:** application and five ratios (credit/income, annuity/income, credit/goods, annuity/credit, income/family), plus an unknown-employment-date indicator.
2. **Level 2:** Level 1 + bureau and bureau_balance: credit counts, debt, arrears, durations and monthly-status fractions.
3. **Level 3:** Level 2 + POS and card: instalment and delinquency history, turnover, payments and credit-card limit utilization.
4. **Level 4:** Level 3 + installments and previous: payment/lateness of observed obligations, amounts and outcomes of previous applications.

Select 3 000 **clients** by stratified sampling with seed 2026 and read all their linked history, including bureau_balance through bureau. Do not independently take the first rows of each table. The technical dataset is saved separately and does not replace the full result. Each stage first saves aggregates with unique `SK_ID_CURR`, followed by left joins; assertions prohibit row multiplication.

In [5]:
pilot = run_preparation(DATA_DIR, PILOT_DIR, REPORTS / 'pilot', sample_size=PILOT_SIZE)
display(pd.DataFrame(pilot['shapes']))
display(pd.DataFrame(pilot['history_coverage']))
print('Pilot: all checks passed:', pilot['checks'])
print('Pilot time, seconds:', pilot['elapsed_seconds'])

application: 3,000 clients, 124 features; 1.6 s


bureau: 2,569 clients, 42 features; 0.9 s


bb: 897 clients, 10 features; 1.8 s


pos: 2,795 clients, 19 features; 1.6 s


cc: 841 clients, 65 features; 2.0 s


installments: 2,822 clients, 21 features; 0.1 s


... (4 outputs left out)

Level 4: 3,000 x 322; 1.2 MiB


,level,rows,features,bytes
0,1,3000,124,362594
1,2,3000,178,599289
2,3,3000,264,858570
3,4,3000,322,1307938


,table,cohort_raw_rows,eligible_rows,eligible_clients,eligible_client_share
0,bureau,14236,14232,2569,0.856333
1,bb,153654,150195,897,0.299000
2,pos,81734,81734,2795,0.931667
3,cc,33501,33501,841,0.280333
4,installments,111993,111932,2822,0.940667
5,previous,13499,13499,2815,0.938333


Pilot: all checks passed: {'unique_applicants': True, 'same_clients_all_levels': True, 'nested_features_and_equal_values': True, 'no_target_ids_gender_age_in_X': True, 'no_train_test_overlap': True, 'repeats_subset_of_development': True, 'no_infinite_values': True, 'missing_history_remains_null': True}
Pilot time, seconds: 52.533


## 5. Full preparation and saved splits

Full preparation starts only after the pilot and resource checks pass. DuckDB is limited to 2 GB and may spill intermediate operations to disk. Historical tables and four X matrices are not all held in memory simultaneously. At least 3 GiB available RAM and 10 GiB free disk are required; otherwise the notebook records the reason and remaining action.

The full population is split once with `train_test_split(test_size=0.20, stratify=TARGET, random_state=42)`. Development is used for later search; test is reserved for final evaluation. Five stratified 90% development subsamples use seeds 42…46. The remaining 10% of a repetition does not become a new test set. Repeated subsamples may overlap, as expected. A common 3-fold assignment is also saved for 28 future searches. Reuse these saved IDs rather than generating separate splits for each level/model.

Fairness attributes are stored separately: original `CODE_GENDER`, `DAYS_BIRTH`, `AGE_YEARS=-DAYS_BIRTH/365.25`, `<25`/`>=25` groups, and an unknown-gender group. Age in years is approximate because calendar dates are unavailable. These attributes, IDs and TARGET are excluded from X. Excluding direct protected attributes does not guarantee the absence of indirect proxies.

In [6]:
available_gib = psutil.virtual_memory().available / 1024**3
disk_gib = shutil.disk_usage(ROOT).free / 1024**3
full = None
full_skip_reason = None
if not RUN_FULL:
    full_skip_reason = 'RUN_FULL=False: set True and run stages 5–7 to finish.'
elif available_gib < 3 or disk_gib < 10:
    full_skip_reason = f'Insufficient resources: RAM={available_gib:.2f} GiB, disk={disk_gib:.2f} GiB. Free resources and repeat stages 5–7.'
else:
    full = run_preparation(DATA_DIR, FULL_DIR, REPORTS / 'full', sample_size=None)
if full is None:
    print('FULL RUN NOT EXECUTED:', full_skip_reason)
else:
    print('FULL RUN COMPLETED:', full['n_applicants'], 'current applications')
    display(pd.DataFrame(full['shapes']))
    display(pd.DataFrame(full['splits']))
    display(pd.DataFrame(full['history_coverage']))
    print('Conflicting obligations excluded:', full['installment_conflicting_obligations_excluded'])

application: 307,511 clients, 124 features; 3.5 s


bureau: 263,475 clients, 42 features; 2.1 s


bb: 92,039 clients, 10 features; 3.8 s


pos: 289,444 clients, 19 features; 3.5 s


cc: 86,905 clients, 65 features; 3.8 s


installments: 291,635 clients, 21 features; 3.4 s


... (6 outputs left out)

,level,rows,features,bytes
0,1,307511,124,22888092
1,2,307511,178,43204706
2,3,307511,264,63449031
3,4,307511,322,103699046


,split,n,target_0,target_1,positive_rate
0,development,246008,226148,19860,0.080729
1,test,61503,56538,4965,0.080728
2,train_seed_42,221407,203533,17874,0.080729
3,train_seed_43,221407,203533,17874,0.080729
4,train_seed_44,221407,203533,17874,0.080729
5,train_seed_45,221407,203533,17874,0.080729
6,train_seed_46,221407,203533,17874,0.080729


,table,cohort_raw_rows,eligible_rows,eligible_clients,eligible_client_share
0,bureau,1465325,1464821,263475,0.856799
1,bb,14701612,14353734,92039,0.299303
2,pos,8543375,8543375,289444,0.941248
3,cc,3227965,3227965,86905,0.282608
4,installments,11591592,11589009,291635,0.948373
5,previous,1413701,1413701,291057,0.946493


Conflicting obligations excluded: 0


## 6. Output checks and file formats

`X_level_N.parquet` contains **features only**, with no identifiers even in its index. All four files are ordered by `SK_ID_CURR`; their exact row mapping is saved in `row_ids.parquet`. `target.parquet` and `fairness.parquet` include IDs for safe alignment. JSON lists preserve feature order and nesting, while `feature_catalog.csv` documents each feature and its SQL formula.

Automatic checks cover unique applications and aggregate keys; identical clients and row counts at every level; equality of shared features; absence of TARGET/ID/gender/age from X; disjoint development/test; repetition IDs contained in development; no infinities; and NULL features for absent history. No client is dropped because history is missing.

An additional check compares pilot and full-data values for **the same clients**, reading only 30 columns at a time. It detects accidental partial-history selection or dependence of deterministic features on sample composition. Tolerance `1e-10` accounts for floating-point summation order during parallel aggregation; no parameters are learned.

In [7]:
crosscheck = compare_pilot_to_full(PILOT_DIR, FULL_DIR) if full is not None else None
print('Pilot/full-data comparison:', crosscheck)
manifest_after = source_manifest(DATA_DIR)
assert manifest_after == manifest_before, 'A source CSV has changed!'
dump_json(REPORTS / 'source_sha256_after.json', manifest_after)
print('All eight source CSV files are unchanged: SHA-256 matches.')
dump_json(REPORTS / 'run_summary.json', {
    'processed_scope': 'full' if full is not None else 'technical_sample_only',
    'audit_seconds': audit['elapsed_seconds'], 'pilot': pilot, 'full': full,
    'full_skip_reason': full_skip_reason, 'pilot_full_comparison': crosscheck,
    'original_csv_sha256_unchanged': True,
    'total_elapsed_seconds': round(time.perf_counter()-RUN_START,3)
})
print('Checks:', (full or pilot)['checks'])

Pilot/full-data comparison: {'pilot_clients': 3000, 'levels_compared': 4, 'same_values': True, 'tolerance': 1e-10}


All eight source CSV files are unchanged: SHA-256 matches.
Checks: {'unique_applicants': True, 'same_clients_all_levels': True, 'nested_features_and_equal_values': True, 'no_target_ids_gender_age_in_X': True, 'no_train_test_overlap': True, 'repeats_subset_of_development': True, 'no_infinite_values': True, 'missing_history_remains_null': True}


## 7. Summary and handoff to modelling

The next cell reports the actual processed volume, dimensions, balance, time and file list. Large data are stored in `data/processed/full/`, intermediate aggregates in `aggregates/`, and split IDs in `splits/`. The pilot is in the adjacent `pilot/` directory. Small reports remain in `reports/`.

To reconstruct X by ID in the next stage:

```python
ids = pd.read_parquet(FULL_DIR / 'row_ids.parquet')['SK_ID_CURR']
X = pd.read_parquet(FULL_DIR / 'X_level_1.parquet')
X.index = pd.Index(ids, name='SK_ID_CURR')  # ID is an index, not an input column
y = pd.read_parquet(FULL_DIR / 'target.parquet').set_index('SK_ID_CURR')['TARGET']
dev_ids = pd.read_parquet(FULL_DIR / 'splits/development_ids.parquet')['SK_ID_CURR']
X_dev, y_dev = X.loc[dev_ids], y.loc[dev_ids]
```

Medians, missing indicators, categories and scaling are fitted **inside each CV training fold**. After search, they are fitted afresh on each 90% training subsample with fixed hyperparameters. Test is not used for these decisions; the fairness file is used only for the specified analysis/correction protocol. Every model at a level uses the same original feature list.

In [8]:
result = full or pilot
print('Actually processed:', result['scope'], '—', result['n_applicants'], 'applications')
display(pd.DataFrame(result['shapes']).rename(columns={'features':'n_features','rows':'n_rows'}))
display(pd.DataFrame([{'TARGET=0': result['target_0'], 'TARGET=1': result['target_1'],
                       'TARGET=1 fraction': result['positive_rate']}]))
run_report = json.loads((REPORTS/'run_summary.json').read_text(encoding='utf-8'))
print('Time, seconds: audit=', audit['elapsed_seconds'], 'pilot=', pilot['elapsed_seconds'],
      'full=', full['elapsed_seconds'] if full else None,
      'whole notebook=', run_report['total_elapsed_seconds'])
generated = []
for directory in [ROOT/'data/processed', REPORTS, ROOT/'notebooks', ROOT/'src', ROOT/'scripts', ROOT/'tests']:
    for path in sorted(directory.rglob('*')):
        if path.is_file() and 'duckdb_tmp' not in str(path) and '__pycache__' not in str(path):
            generated.append({'file':str(path.relative_to(ROOT)), 'bytes':path.stat().st_size})
for name in ['.gitignore','requirements.txt']:
    path = ROOT/name
    generated.append({'file':name, 'bytes':path.stat().st_size})
inventory = pd.DataFrame(generated)
inventory.to_csv(REPORTS/'generated_files.csv', index=False)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120):
    display(inventory)
if full_skip_reason:
    print(full_skip_reason)

Actually processed: full — 307511 applications


,level,n_rows,n_features,bytes
0,1,307511,124,22888092
1,2,307511,178,43204706
2,3,307511,264,63449031
3,4,307511,322,103699046


,TARGET=0,TARGET=1,TARGET=1 fraction
0,282686,24825,0.080729


Time, seconds: audit= 97.229 pilot= 52.533 full= 151.576 whole notebook= 309.197


,file,bytes
0,data\processed\full\aggregates\application.parquet,23202644
1,data\processed\full\aggregates\bb.parquet,1531011
2,data\processed\full\aggregates\bureau.parquet,17255656
3,data\processed\full\aggregates\bureau_id_map.parquet,3310441
4,data\processed\full\aggregates\cc.parquet,9953044
5,data\processed\full\aggregates\installment_obligations.parquet,196304476
6,data\processed\full\aggregates\installments.parquet,13833955
7,data\processed\full\aggregates\pos.parquet,5036909
8,data\processed\full\aggregates\previous.parquet,25911140
9,data\processed\full\fairness.parquet,3006562


## 8. Additional data-quality checks before modeling

This section uses the **existing** `development_ids.parquet`, four X files, and feature lists. Splits are not recreated, and features are not overwritten. SHA-256 hashes of all prepared files are checked before and after. Distributions, percentiles, and semantic constraints are analyzed on **development** only; test data participate only in mechanical preservation checks and the requested breakdown of 2,905 rows with incomplete keys. Test data are not used for outlier decisions.

Results are written to `reports/quality/`. Counts of **raw history values**, **clients**, and **aggregate cells** are reported separately because these are different counting units.

In [9]:
from pathlib import Path
import sys, os, json, time
import pandas as pd
from IPython.display import display

QA_ROOT=Path.cwd()
if QA_ROOT.name=='notebooks':
    QA_ROOT=QA_ROOT.parent
sys.path.insert(0,str(QA_ROOT))
from src.data_quality import (artifact_manifest, development_profile, semantic_checks, sentinel_checks)
from src.payment_quality import audit_payment_keys
from src.fold_preprocessing import make_numeric_pipeline, numeric_pipeline_smoke_check
from src.data_preparation import dump_json, source_manifest

QA_DATA=Path(os.environ.get('HOME_CREDIT_DATA_DIR',r'C:\Users\askat\Study\RMT\home-credit-default-risk'))
QA_PREP=QA_ROOT/'data/processed/full'
QA_REPORT=QA_ROOT/'reports/quality'
QA_REPORT.mkdir(parents=True,exist_ok=True)
QA_STARTED=time.perf_counter()
qa_manifest_before=artifact_manifest(QA_PREP)
qa_sources_before=source_manifest(QA_DATA)
dump_json(QA_REPORT/'prepared_sha256_before.json',qa_manifest_before)
print('Existing prepared files protected by SHA-256:',len(qa_manifest_before))

Existing prepared files protected by SHA-256: 29


### 8.1. Numeric distributions in development

Exact linearly interpolated P1, P25, median, P75, P99, and P99.9 are computed from nonmissing values; missingness, min/max, zeros, negative values, and infinities are counted separately. Computing a median for the **report** neither imputes data nor fits a transformation. Data are read in batches of 24 columns to avoid holding all levels in memory.

The full table is `numeric_development_summary.csv`; `first_level` and `levels` indicate feature membership. Shared features are profiled once. Entirely missing columns, a single unique **observed** value with missingness, and truly constant columns without missingness are distinguished. No columns are removed based on this global summary.

In [10]:
qa_profile=development_profile(QA_PREP,QA_REPORT)
display(pd.DataFrame(qa_profile['levels']))
numeric_summary=pd.read_csv(QA_REPORT/'numeric_development_summary.csv')
print('Full summary:',QA_REPORT/'numeric_development_summary.csv')
display(numeric_summary.sort_values('missing_fraction',ascending=False)[
    ['feature','first_level','missing','missing_fraction','min','p01','p25','median','p75','p99','p999','max','zeros','negatives']].head(25))
print('Empty and constant columns (all types):')
display(pd.read_csv(QA_REPORT/'empty_and_constant_columns.csv'))
print('Entirely missing numeric columns:',qa_profile['all_missing_numeric'])
print('Truly constant numeric columns without missingness:',qa_profile['constant_fully_numeric'])

,level,rows,total_features,numeric_features,all_missing_numeric,constant_observed_numeric
0,1,246008,124,109,0,0
1,2,246008,178,163,0,0
2,3,246008,264,249,0,2
3,4,246008,322,307,0,2


Full summary: reports\quality\numeric_development_summary.csv


,feature,first_level,missing,missing_fraction,min,p01,p25,median,p75,p99,p999,max,zeros,negatives
211,CC_AMT_PAYMENT_CURRENT_MAX,3,197059,0.801027,0.000000,930.1428,22500.000000,65250.000000,157500.000000,748601.411400,1.028514e+06,4.289207e+06,103,0
210,CC_AMT_PAYMENT_CURRENT_MIN,3,197059,0.801027,0.000000,0.0000,0.000000,0.000000,597.375000,27000.000000,1.002379e+05,1.125000e+06,30774,0
209,CC_AMT_PAYMENT_CURRENT_AVG,3,197059,0.801027,0.000000,248.8032,4657.500000,9855.000000,21416.544783,122128.860600,3.229143e+05,1.125000e+06,103,0
224,CC_CNT_DRAWINGS_ATM_CURRENT_AVG,3,196989,0.800742,0.000000,0.0000,0.063158,0.272727,0.733333,3.857143,8.000000e+00,3.500000e+01,7884,0
226,CC_CNT_DRAWINGS_ATM_CURRENT_MAX,3,196989,0.800742,0.000000,0.0000,2.000000,3.000000,6.000000,17.000000,2.798200e+01,5.100000e+01,7884,0
230,CC_CNT_DRAWINGS_OTHER_CURRENT_AVG,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,0.166667,6.666667e-01,2.000000e+00,44467,0
231,CC_CNT_DRAWINGS_OTHER_CURRENT_MIN,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2.000000e+00,49016,0
232,CC_CNT_DRAWINGS_OTHER_CURRENT_MAX,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,2.000000,4.000000e+00,1.200000e+01,44467,0
233,CC_CNT_DRAWINGS_POS_CURRENT_AVG,3,196989,0.800742,0.000000,0.0000,0.000000,0.043478,1.434783,20.544196,4.290173e+01,1.182500e+02,21640,0
234,CC_CNT_DRAWINGS_POS_CURRENT_MIN,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,5.000000,2.000000e+01,7.800000e+01,46809,0


Empty and constant columns (all types):


,feature,first_level,levels,dtype,n_development,missing,missing_fraction,n_unique_observed,all_missing,constant_among_observed,constant_including_missing,constant_value
0,CC_SK_DPD_MIN,3,"3,4",float64,246008,176325,0.716745,1,False,True,False,0.0
1,CC_SK_DPD_DEF_MIN,3,"3,4",float64,246008,176325,0.716745,1,False,True,False,0.0


Entirely missing numeric columns: []
Truly constant numeric columns without missingness: []


### 8.2. Semantic constraints and proposed actions

Strict rules require positive current income/credit, nonnegative nominal amounts, integer family/child counts, and `*_SHARE` fractions in `[0,1]`. A missing value is neither negative nor zero. Absence of a sign violation does not establish that an extreme value is accurate.

Amount ratios **need not fall within `[0,1]`**: overpayments, exceeded credit limits, and differences between requested and approved credit are possible. The dictionary does not define comparable periods for income and annuity, so `annuity/income > 1` alone does not justify correction. Negative balances and cash movements can represent overpayments or reversals; no blanket rule removes all AMT < 0 values.

For each rule, `semantic_source_checks.csv` provides the SQL condition, rationale, action, status, and number of affected raw values/clients. The specific proposal for negative down payments is to replace them with NULL **before** aggregation after confirming the sign convention; the exact number of changed aggregates is computed, but the replacement is not applied now. Ambiguous limits/movements require source clarification rather than unsupported correction.

Rare large values are retained. P99/P99.9 describe the tail and are not removal, clipping, or winsorization thresholds.

In [11]:
qa_semantics=semantic_checks(QA_DATA,QA_PREP,QA_REPORT)
display(pd.read_csv(QA_REPORT/'semantic_source_checks.csv')[
    ['source','field','condition','affected_values','affected_clients','classification','proposed_action_ru','reason_ru']])
display(pd.read_csv(QA_REPORT/'ratio_checks_development.csv'))
focus=['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE','CNT_FAM_MEMBERS','CNT_CHILDREN',
       'APP_CREDIT_TO_INCOME','APP_ANNUITY_TO_INCOME','APP_CREDIT_TO_GOODS','APP_ANNUITY_TO_CREDIT','APP_INCOME_PER_MEMBER']
display(numeric_summary[numeric_summary.feature.isin(focus)][
    ['feature','missing','min','p01','p25','median','p75','p99','p999','max','zeros','negatives']])
print('Estimated impact of the proposed replacement; not applied:')
display(pd.DataFrame([qa_semantics['proposed_fix']]))
print('Income tail, retained unchanged:',qa_semantics['income_tail'])

,source,field,condition,affected_values,affected_clients,classification,proposed_action_ru,reason_ru
0,application,AMT_INCOME_TOTAL,AMT_INCOME_TOTAL<=0,0,0,hard,NULL for income <=0; currently unnecessary,Positive income is required to interpret...
1,application,AMT_CREDIT,AMT_CREDIT<=0,0,0,hard,NULL for credit <=0; currently unnecessary,The current loan amount must be positive...
2,application,AMT_ANNUITY,AMT_ANNUITY<0,0,0,hard,NULL for a negative annuity; currently not needed...,A negative regular obligation amount...
3,application,AMT_GOODS_PRICE,AMT_GOODS_PRICE<0,0,0,hard,NULL for negative value; currently not needed...,The cost of the purchased goods is nonnegative.
4,application,CNT_FAM_MEMBERS,CNT_FAM_MEMBERS<1 OR CNT_FAM_MEMBERS<>floor(CN...,0,0,hard,NULL for family count <1 or noninteger; currently...,Family size includes the applicant and is...
5,application,CNT_CHILDREN,CNT_CHILDREN<0 OR CNT_CHILDREN<>floor(CNT_CHIL...,0,0,hard,NULL for a negative/fractional count; currently...,The number of children is a nonnegative integer.
6,application,CNT_FAM_MEMBERS/CNT_CHILDREN,CNT_FAM_MEMBERS<=CNT_CHILDREN,0,0,review,Check family composition; do not correct automatically...,The condition assumes inclusion of the applicant and children...
7,previous,AMT_APPLICATION,AMT_APPLICATION<0,0,0,hard,NULL for negative requested amount; currently...,The nominal requested amount is nonnegative.
8,previous,AMT_CREDIT,AMT_CREDIT<0,0,0,hard,NULL for negative approved amount; currently...,The nominal credit amount is nonnegative.
9,previous,AMT_APPLICATION,AMT_APPLICATION=0,260183,109383,allowed,Retain,Cancelled/rejected applications and other product types...


,feature,missing,negative,above_one,infinite,rule_violations,rule_ru,applied_correction
0,APP_CREDIT_TO_INCOME,0,0,233085,0,0,A positive amount ratio need not be <...,False
1,APP_ANNUITY_TO_INCOME,10,0,25,0,0,A positive amount ratio need not be <...,False
2,APP_CREDIT_TO_GOODS,221,0,158985,0,0,A positive amount ratio need not be <...,False
3,APP_ANNUITY_TO_CREDIT,10,0,0,0,0,A positive amount ratio need not be <...,False
4,APP_INCOME_PER_MEMBER,2,0,246006,0,0,A positive amount ratio need not be <...,False
5,BUREAU_ACTIVE_SHARE,35256,0,0,0,0,"A proportion must lie in [0,1].",False
6,BUREAU_CLOSED_SHARE,35256,0,0,0,0,"A proportion must lie in [0,1].",False
7,BUREAU_DEBT_TO_CREDIT,41932,1002,1244,0,0,Negative debt is possible after overpayment;...,False
8,BB_DELINQUENT_SHARE,172496,0,0,0,0,"A proportion must lie in [0,1].",False
9,BB_SEVERE_DELINQUENT_SHARE,172496,0,0,0,0,"A proportion must lie in [0,1].",False


,feature,missing,min,p01,p25,median,p75,p99,p999,max,zeros,negatives
0,CNT_CHILDREN,0,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000e+00,4.000000e+00,1.900000e+01,172232,0
1,AMT_INCOME_TOTAL,0,25650.000000,45000.000000,112500.000000,147600.000000,202500.000000,4.725000e+05,9.000000e+05,1.170000e+08,0,0
2,AMT_CREDIT,0,45000.000000,76410.000000,270000.000000,514777.500000,808650.000000,1.862802e+06,2.517300e+06,4.050000e+06,0,0
3,AMT_ANNUITY,10,1615.500000,6187.500000,16510.500000,24903.000000,34596.000000,7.003350e+04,1.110018e+05,2.580255e+05,0,0
4,AMT_GOODS_PRICE,221,40500.000000,67500.000000,238500.000000,450000.000000,679500.000000,1.800000e+06,2.250000e+06,4.050000e+06,0,0
16,CNT_FAM_MEMBERS,2,1.000000,1.000000,2.000000,2.000000,3.000000,5.000000e+00,6.000000e+00,2.000000e+01,0,0
104,APP_CREDIT_TO_INCOME,0,0.004808,0.595789,2.018667,3.268948,5.168514,1.303571e+01,1.930662e+01,4.922720e+01,0,0
105,APP_ANNUITY_TO_INCOME,10,0.000224,0.039371,0.114583,0.162833,0.229000,4.838502e-01,7.262502e-01,1.570600e+00,0,0
106,APP_CREDIT_TO_GOODS,221,0.150000,1.000000,1.000000,1.118800,1.198000,1.475200e+00,1.528000e+00,6.000000e+00,0,0
107,APP_ANNUITY_TO_CREDIT,10,0.022073,0.026459,0.036900,0.050000,0.063910,1.144990e-01,1.201000e-01,1.244296e-01,0,0


Estimated impact of the proposed replacement; not applied:


,proposal,raw_values,changed_aggregate_cells,changed_clients,per_feature_changed_cells,applied,reason
0,previous AMT_DOWN_PAYMENT<0 -> NULL before agg...,1,2,1,"{'avg': 1, 'min': 1, 'max': 0}",False,First confirm that -0.9 is not...


Income tail, retained unchanged: {'p99': 472500.0, 'p999': 900000.0, 'max': 117000000.0, 'above_p999': 222, 'removed': 0, 'winsorized': 0, 'note': 'Quantiles were computed on development only and are not cleaning thresholds.'}


### 8.3. Sentinel values: source → rule → saved result

For `DAYS_EMPLOYED=365243`, checks verify both absence of the number from X and agreement with the source for **every application**: the value becomes NULL, `APP_EMPLOYMENT_DATE_UNKNOWN=1`, and other valid dates remain unchanged. Other application dates describing actual events are checked similarly.

In previous applications, `365243` is replaced with NULL in the scheduled date fields used: `DAYS_FIRST_DUE` and `DAYS_LAST_DUE_1ST_VERSION`. Their AVG/MIN/MAX are recomputed from cleaned rows and compared with saved aggregates for all prepared applications. `DAYS_FIRST_DRAWING`, `DAYS_LAST_DUE`, and `DAYS_TERMINATION` are excluded from X entirely: 'after' means no feature, rather than a count of imputed values. Positive scheduled dates are retained.

Before/after NULL counts use the same set of **raw rows**, because comparing raw NULL counts with client-aggregate counts would be invalid. Aggregate verification is saved separately. `placeholder_before` always refers only to `365243`. The auxiliary `positive_before` counts all positive values, including the sentinel, for application dates describing actual events, but only positive nonsentinel values for scheduled previous/bureau dates: these fields have different semantic rules. Mechanical checks of final temporal features do not tune parameters on test data.

In [12]:
qa_sentinels=sentinel_checks(QA_DATA,QA_PREP,QA_REPORT)
display(pd.read_csv(QA_REPORT/'sentinels_before_after.csv')[
    ['source','field','scope','source_rows','placeholder_before','null_before','positive_before',
     'null_after_same_rows','placeholder_after_same_rows','saved_feature_status']])
display(pd.read_csv(QA_REPORT/'saved_date_aggregate_checks.csv'))
print('Saved sentinel checks:',qa_sentinels)

,source,field,scope,source_rows,placeholder_before,null_before,positive_before,null_after_same_rows,placeholder_after_same_rows,saved_feature_status
0,application,DAYS_EMPLOYED,all_source,307511,55374,0,55374,55374.0,0.0,included
1,application,DAYS_REGISTRATION,all_source,307511,0,0,0,0.0,0.0,included
2,application,DAYS_ID_PUBLISH,all_source,307511,0,0,0,0.0,0.0,included
3,application,DAYS_LAST_PHONE_CHANGE,all_source,307511,0,1,0,1.0,0.0,included
4,application,DAYS_EMPLOYED,development,246008,44143,0,44143,44143.0,0.0,included
5,application,DAYS_REGISTRATION,development,246008,0,0,0,0.0,0.0,included
6,application,DAYS_ID_PUBLISH,development,246008,0,0,0,0.0,0.0,included
7,application,DAYS_LAST_PHONE_CHANGE,development,246008,0,1,0,1.0,0.0,included
8,previous,DAYS_FIRST_DRAWING,all_source,1670214,934444,673065,0,NaN,NaN,excluded
9,previous,DAYS_FIRST_DUE,all_source,1670214,40645,673065,0,713710.0,0.0,included


,feature,prepared_clients_with_history,mismatches_to_cleaned_history,positive_saved_values,remaining_365243
0,PREVIOUS_DAYS_FIRST_DUE_AVG,291057,0,0,0
1,PREVIOUS_DAYS_FIRST_DUE_MIN,291057,0,0,0
2,PREVIOUS_DAYS_FIRST_DUE_MAX,291057,0,0,0
3,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_AVG,291057,0,46597,0
4,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MIN,291057,0,29608,0
5,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MAX,291057,0,147954,0


Saved sentinel checks: {'source_before_after_rows': 20, 'application_rule_mismatches': 0, 'employment_indicator_mismatches': 0, 'saved_previous_aggregate_mismatches': 0, 'development_saved_date_placeholders': 0, 'all_prepared_saved_date_placeholders': 0, 'note': 'NULL counts before/after refer to source rows; aggregate checks are saved separately. For excluded fields, after=not applicable.'}


### 8.4. Incomplete keys in 2,905 installments_payments rows

The complete candidate key checked for a payment row is:
`(SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER, DAYS_ENTRY_PAYMENT)`.
`SK_ID_CURR` additionally identifies the owner and links to the current application. This is **not a guaranteed unique payment-event ID**: several partial payments may share the same date and candidate key.

The installment key used in preparation is:
`(SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER)`.
Actual payment date is not part of this key. Partial-payment amounts are summed exactly, while the scheduled amount is counted once. Full duplicates are diagnosed; repeated candidate keys do not trigger deletion.

The report shows missingness in all eight fields, breakdown by the existing splits, other payments for the same installment, and client preservation. A row without an actual payment date does not satisfy `<0`; an unknown amount is not filled with zero. Such a row establishes neither payment nor nonpayment reliably. Final checks verify preservation of the **number of raw payment rows** and the **exact payment total** in aggregated installments.

In [13]:
qa_payments=audit_payment_keys(QA_DATA,QA_PREP,QA_REPORT)
display(pd.read_csv(QA_REPORT/'installments_incomplete_key_fields.csv'))
display(pd.read_csv(QA_REPORT/'installments_incomplete_key_patterns.csv'))
display(pd.DataFrame(qa_payments['by_split']))
display(pd.DataFrame(qa_payments['obligation_recovery']))
display(pd.DataFrame(qa_payments['client_retention']))
display(pd.DataFrame([qa_payments['partial_payment_preservation']]).T)
print('Incomplete rows:',qa_payments['missing_key_rows'],'Missing fields:',qa_payments['null_fields'])

,column,part_of_candidate_key,part_of_obligation_key,null_rows,non_null_rows
0,SK_ID_PREV,True,True,0,2905
1,SK_ID_CURR,False,True,0,2905
2,NUM_INSTALMENT_VERSION,True,True,0,2905
3,NUM_INSTALMENT_NUMBER,True,True,0,2905
4,DAYS_INSTALMENT,False,False,0,2905
5,DAYS_ENTRY_PAYMENT,True,False,2905,0
6,AMT_INSTALMENT,False,False,0,2905
7,AMT_PAYMENT,False,False,2905,0


,null_pattern,rows
0,DAYS_ENTRY_PAYMENT; AMT_PAYMENT,2905


,split,incomplete_rows,clients,rows_eligible,rows_excluded
0,development,2091,861,0,2091
1,outside_labeled_applications,322,174,0,322
2,test,492,214,0,492


,split,affected_obligations,obligations_with_other_valid_payments,obligations_without_valid_payments,other_valid_payment_rows,obligations_in_saved_aggregation
0,development,2091,0,2091,0.0,0
1,outside_labeled_applications,322,0,322,0.0,0
2,test,492,0,492,0.0,0


,split,affected_clients,retained_current_applicants,clients_with_other_saved_installment_history
0,development,861,861,856
1,outside_labeled_applications,174,0,0
2,test,214,214,211


,0
raw_rows,13605401
exact_duplicate_row_surplus,0
repeated_candidate_key_groups,770
candidate_key_duplicate_surplus,770
rows_in_repeated_candidate_keys,1540
max_rows_per_candidate_key,2
repeated_keys_with_distinct_payment_amounts,770
eligible_labeled_payment_rows,11589009
eligible_labeled_nonmissing_payment_amounts,11589009
saved_obligations,11024044


Incomplete rows: 2905 Missing fields: ['DAYS_ENTRY_PAYMENT', 'AMT_PAYMENT']


### 8.5. An entirely missing column within a training fold

Even if no column is entirely missing across development, this can occur in an individual training fold or 90% repeat. The next stage uses the prepared factory `src.fold_preprocessing.make_numeric_pipeline(scale=False)`, which returns an **unfitted** object.

Numeric branch: `SimpleImputer(strategy='median', keep_empty_features=True)`. If a column is entirely missing in the **current training set**, no median exists: the predefined technical fallback is **0**, and the column is retained. Observed validation/test values in that column remain unchanged; only missing values are filled. This is not an estimate from development/test. With no observed training values, the model has no basis for estimating the variable's effect.

In parallel, `MissingIndicator(features='all')` creates an original-missingness indicator for every numeric column, including missingness first seen in validation. Numeric output width is stable at **2 × p**, regardless of which columns were missing in training. With `scale=True`, StandardScaler is fitted only on the value branch within training; indicators are not scaled. After fallback for an entirely missing column, the scaler has mean=0 and scale=1.

Categorical encoding and optional feature selection remain in the full Pipeline at the next stage. The original feature lists for all four levels are preserved. The factory must be included in ColumnTransformer and Pipeline **before passing to CV**, rather than calling fit_transform on all development data beforehand. Only a check on three synthetic columns is run here; Home Credit data undergo no fit, imputation, scaling, or encoding.

In [14]:
unfitted_numeric_pipeline=make_numeric_pipeline(scale=False)
qa_pipeline=numeric_pipeline_smoke_check(QA_REPORT/'fold_pipeline_checks.json')
print('Demonstration on synthetic data only:')
display(pd.DataFrame([qa_pipeline]))

Demonstration on synthetic data only:


,scope,sklearn_version,all_checks_passed,raw_numeric_columns,transformed_columns,empty_train_column_fallback,median_for_partly_missing_train_column,validation_observed_value_in_empty_train_column_preserved,new_validation_missing_value_has_indicator,width_is_twice_numeric_column_count,feature_names,fit_policy_ru,fallback_policy_ru,category_policy_ru
0,synthetic_only_no_fit_on_home_credit,1.7.2,True,3,6,0,2,True,True,True,"[values__empty_in_train, values__partly_missin...",fit only on each CV training fold or 90%...,"For an entirely missing training column, a fixed...",Learned categorical encoding is a separate branch...


### 8.6. Audit summary and open questions

**Already corrected during preparation:** date sentinels are replaced with NULL or fields are excluded; unavailable actual future/undated payments are filtered out; division by a nonpositive denominator yields NULL; partial payments are combined without losing amounts or repeating the installment nominal amount. Decimal arithmetic prevents false underpayment at exact equality.

**Retained:** rare large incomes/amounts, ratios >1, positive contractual dates, zero amounts for rejected/cancelled previous applications, negative balances/movements pending clarification of their accounting meaning, and two features with a constant observed zero plus missingness. Data quality does not imply automatic distribution normalization.

**Kept inside the Pipeline:** training-fold medians, entirely missing training-column handling with fallback=0 and indicators, scaling when needed, learned encoding, and selection. None of these stages is fitted on the full Home Credit data.

**Open questions:** confirmation of the 117 million income value and income units/periods; sign conventions for negative down payments, withdrawals, limits, and balances; unavoidable incomplete information for installments missing payment date/amount; exact availability times of updated historical values. Where dictionary information is insufficient, ambiguous values are flagged for review rather than silently corrected.

The final cell confirms preservation of all X files, feature lists, target, fairness data, IDs, and aggregates; raw CSVs are also verified by SHA-256.

In [15]:
qa_manifest_after=artifact_manifest(QA_PREP)
assert qa_manifest_before==qa_manifest_after,'Prepared data or splits changed!'
assert qa_sources_before==source_manifest(QA_DATA),'Raw CSVs changed!'
dump_json(QA_REPORT/'prepared_sha256_after.json',qa_manifest_after)
qa_result={'scope':'development_quality_audit_existing_prepared_data',
           'profile':qa_profile,'semantics':qa_semantics,'sentinels':qa_sentinels,
           'payments':qa_payments,'pipeline':qa_pipeline,
           'prepared_files_sha256_unchanged':True,'source_csv_sha256_unchanged':True,
           'X_and_split_files_modified':0,'global_fit_on_home_credit':False,
           'elapsed_seconds':round(time.perf_counter()-QA_STARTED,3)}
dump_json(QA_REPORT/'quality_summary.json',qa_result)
print('Development:',qa_profile['development_rows'],'numeric features:',qa_profile['numeric_features'])
print('Strict violations:',qa_semantics['hard_rule_violations'],'ratio-rule violations:',qa_semantics['ratio_rule_violations'])
print('X, levels, splits, and sources: SHA-256 hashes unchanged. This audit did not modify values.')
print('Additional audit time, seconds:',qa_result['elapsed_seconds'])
quality_files=pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size} for p in sorted(QA_REPORT.iterdir()) if p.is_file()])
display(quality_files)

Development: 246008 numeric features: 307
Strict violations: 0 ratio-rule violations: 0
X, levels, splits, and sources: SHA-256 hashes unchanged. This audit did not modify values.
Additional audit time, seconds: 165.255


,file,bytes
0,column_development_diagnostics.csv,30381
1,development_levels.csv,181
2,empty_and_constant_columns.csv,350
3,financial_and_ratio_distribution.csv,5361
4,fold_pipeline_checks.json,1246
5,income_tail_review.json,264
6,installments_incomplete_client_retention.csv,181
7,installments_incomplete_key_by_split.csv,155
8,installments_incomplete_key_fields.csv,361
9,installments_incomplete_key_patterns.csv,60
